# YOLOv8 + VLM: распознавание полей паспорта

Ноутбук:

1. загружает обученную модель `runs/detect/idd_passport_fields_yolov8n/weights/best.pt`;
2. делает предсказания на validation-выборке датасета `datasets/idd_passport`;
3. вырезает найденные YOLO области и отправляет их в LM Studio (OpenAI-совместимый API);
4. собирает распознанный текст и сохраняет JSON **рядом с ноутбуком**.

**Формат результата** (`vlm_predictions.json`):

```json
[
  {
    "01_jpg.rf.xxxx.jpg": {
      "type": "P",
      "surname": "IOANNOU",
      "mrz_line0": "P<GRC..."
    }
  }
]
```

**Перед запуском:** в LM Studio должна быть загружена модель `qwen/qwen3.8-27b`,
а локальный сервер — включён на `http://192.168.1.150:1234`.

> Запросы идут по одному кропу. На 120 validation-изображениях получается
> около 2 200 кропов, поэтому полный прогон может занять несколько часов
> (скорость зависит от LM Studio и GPU). Результат периодически сохраняется,
> так что прогресс не потеряется. Для быстрой проверки задайте `MAX_IMAGES = 2`.


In [ ]:
from __future__ import annotations

import base64
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import cv2
import pandas as pd
import requests
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO

# --- пути -------------------------------------------------------------------
NB_DIR = Path.cwd()
if not (NB_DIR / "train_yolov8.ipynb").exists() and (NB_DIR / "notebooks" / "train_yolov8.ipynb").exists():
    NB_DIR = NB_DIR / "notebooks"
ROOT = NB_DIR.parent if (NB_DIR.parent / "runs").exists() else NB_DIR

RUN_NAME = "idd_passport_fields_yolov8n"
WEIGHTS = ROOT / "runs" / "detect" / RUN_NAME / "weights" / "best.pt"
DATASET_DIR = ROOT / "datasets" / "idd_passport"
DATA_YAML = DATASET_DIR / "data.yaml"
VAL_IMAGES_DIR = DATASET_DIR / "images" / "val"
OUTPUT_PATH = NB_DIR / "vlm_predictions.json"

# --- YOLO -------------------------------------------------------------------
IMGSZ = 640
CONF = 0.25
IOU = 0.45
DEVICE = 0 if torch.cuda.is_available() else "cpu"
SKIP_CLASSES = {"photo", "face"}
MAX_IMAGES = None  # None — все validation-изображения; для проверки поставьте 2

# --- VLM (LM Studio) --------------------------------------------------------
VLM_BASE_URL = "http://192.168.1.150:1234/v1"
VLM_MODEL = "qwen/qwen3.8-27b"
REQUEST_TIMEOUT = 180
MAX_RETRIES = 3
MAX_WORKERS = 8  # параллельные запросы к LM Studio
MAX_TOKENS = 256
SAVE_EVERY = 50  # как часто сохранять промежуточный JSON

# --- кроп -------------------------------------------------------------------
CROP_PAD = 4
CROP_MIN_HEIGHT = 48
CROP_MAX_SCALE = 3.0

PROMPT_TEMPLATE = (
    "Ты OCR-система для паспортных документов. На изображении — фрагмент документа, "
    "поле «{field}». Распознай весь видимый текст и верни ТОЛЬКО распознанный текст, "
    "без пояснений, кавычек, markdown и подписей. Сохраняй переносы строк. "
    "Символы MRZ '<' сохраняй как есть. Если текста нет, верни пустую строку."
)

print(f"Ноутбук:    {NB_DIR}")
print(f"Веса:       {WEIGHTS}")
print(f"Валидация:  {VAL_IMAGES_DIR}")
print(f"Результат:  {OUTPUT_PATH}")
print(f"Устройство: {DEVICE}")


## 1. Загрузка модели и проверка данных


In [ ]:
assert WEIGHTS.exists(), f"Не найдены веса: {WEIGHTS}"
assert DATA_YAML.exists(), f"Не найден data.yaml: {DATA_YAML}"

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
NAMES = {int(k): v for k, v in cfg["names"].items()}

val_images = sorted(VAL_IMAGES_DIR.glob("*.jpg"))
if MAX_IMAGES is not None:
    val_images = val_images[:MAX_IMAGES]

assert val_images, f"В {VAL_IMAGES_DIR} нет изображений"
print(f"Классов: {len(NAMES)}")
print(f"Validation-изображений к обработке: {len(val_images)}")

model = YOLO(str(WEIGHTS))


## 2. Проверка LM Studio

Убеждаемся, что сервер доступен и нужная модель загружена.


In [ ]:
def check_vlm_server():
    url = f"{VLM_BASE_URL}/models"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    model_ids = [item["id"] for item in response.json().get("data", [])]
    print(f"LM Studio доступен: {url}")
    print("Модели:", model_ids)
    if VLM_MODEL not in model_ids:
        raise RuntimeError(
            f"Модель {VLM_MODEL!r} не найдена. Загрузите её в LM Studio или измените VLM_MODEL."
        )


check_vlm_server()


## 3. Функции кропа и OCR

- кроп обрезается по рамке YOLO с небольшим отступом;
- мелкие кропы (например, MRZ) увеличиваются;
- кроп кодируется в JPEG и передаётся в `image_url` как data URI;
- ответ модели очищается от markdown и служебных подписей.


In [ ]:
_thread_local = threading.local()


def get_session() -> requests.Session:
    session = getattr(_thread_local, "session", None)
    if session is None:
        session = requests.Session()
        _thread_local.session = session
    return session


def crop_detection(image, xyxy, pad: int = CROP_PAD):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = (int(round(value)) for value in xyxy)
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)
    if x2 <= x1 or y2 <= y1:
        return None

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    crop_h, crop_w = crop.shape[:2]
    scale = min(max(1.0, CROP_MIN_HEIGHT / max(1, crop_h)), CROP_MAX_SCALE)
    if scale > 1.0:
        crop = cv2.resize(
            crop,
            (max(1, int(crop_w * scale)), max(1, int(crop_h * scale))),
            interpolation=cv2.INTER_CUBIC,
        )
    return crop


def encode_crop(crop) -> str:
    ok, buffer = cv2.imencode(".jpg", crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise RuntimeError("Не удалось закодировать кроп в JPEG")
    return base64.b64encode(buffer.tobytes()).decode("ascii")


def clean_text(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```(?:text|json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip().strip('"').strip("'").strip()
    text = re.sub(r"^(?:текст|ocr|распознанный текст)\s*[:：]\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def recognize_crop(crop, field: str) -> str:
    prompt = PROMPT_TEMPLATE.format(field=field)
    image_url = f"data:image/jpeg;base64,{encode_crop(crop)}"
    request_payload = {
        "model": VLM_MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }
        ],
        "temperature": 0,
        "max_tokens": MAX_TOKENS,
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = get_session().post(
                f"{VLM_BASE_URL}/chat/completions",
                json=request_payload,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"].get("content") or ""
            return clean_text(content)
        except (requests.RequestException, KeyError, IndexError, TypeError) as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(min(2 ** attempt, 10))

    print(f"[WARN] {field}: не удалось распознать ({last_error})")
    return ""


## 4. Предсказания YOLO и подготовка кропов

Для каждого изображения берём по одной (самой уверенной) детекции каждого класса:
формула результата предполагает соответствие `класс -> текст`.
Классы `photo` и `face` пропускаются, так как текста в них нет.


In [ ]:
predictions = model.predict(
    source=[str(path) for path in val_images],
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    device=DEVICE,
    verbose=False,
)

tasks = []
image_order = []

for result in predictions:
    image_name = Path(result.path).name
    image_order.append(image_name)

    best_by_class = {}
    for box in result.boxes:
        field = NAMES[int(box.cls)]
        if field in SKIP_CLASSES:
            continue
        confidence = float(box.conf)
        previous = best_by_class.get(field)
        if previous is None or confidence > previous[0]:
            best_by_class[field] = (confidence, box.xyxy[0].tolist())

    for field, (confidence, xyxy) in best_by_class.items():
        crop = crop_detection(result.orig_img, xyxy)
        if crop is not None:
            tasks.append((image_name, field, crop, confidence))

print(f"Изображений: {len(predictions)}")
print(f"Кропов к распознаванию: {len(tasks)}")
print(f"В среднем на изображение: {len(tasks) / max(1, len(predictions)):.1f}")


## 5. Распознавание текста через VLM

Запросы выполняются параллельно (`MAX_WORKERS`). Для каждого кропа сохраняется
только текст; при ошибке после `MAX_RETRIES` попыток записывается пустая строка.


In [ ]:
results_by_image = {name: {} for name in image_order}
records = []
errors = []


def save_results():
    payload = [{name: results_by_image[name]} for name in image_order]
    OUTPUT_PATH.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return payload


def run_task(task):
    image_name, field, crop, confidence = task
    text = recognize_crop(crop, field)
    return image_name, field, confidence, text


started = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(run_task, task) for task in tasks]
    for done, future in enumerate(as_completed(futures), start=1):
        try:
            image_name, field, confidence, text = future.result()
        except Exception as exc:  # noqa: BLE001
            errors.append(str(exc))
            continue

        results_by_image[image_name][field] = text
        records.append(
            {
                "image": image_name,
                "class": field,
                "confidence": round(confidence, 3),
                "text": text,
            }
        )

        if done % 25 == 0 or done == len(tasks):
            elapsed = time.time() - started
            speed = done / max(elapsed, 1e-9)
            remaining = (len(tasks) - done) / max(speed, 1e-9)
            print(
                f"[{done}/{len(tasks)}] {speed:.2f} запр./с, "
                f"осталось ~{remaining / 60:.1f} мин"
            )

        if done % SAVE_EVERY == 0:
            save_results()

result_payload = save_results()

print(f"\nГотово за {(time.time() - started) / 60:.1f} мин")
print(f"Ошибок: {len(errors)}")
print(f"Сохранено: {OUTPUT_PATH}")


## 6. Результаты


In [ ]:
print(f"Файл: {OUTPUT_PATH}")
print(f"Изображений в JSON: {len(result_payload)}")

recognized = sum(1 for record in records if record["text"])
print(f"Непустых распознаваний: {recognized}/{len(records)}")

if result_payload:
    sample_name = image_order[0]
    print("\nПример записи:")
    print(json.dumps({sample_name: results_by_image[sample_name]}, ensure_ascii=False, indent=2))

if records:
    display(pd.DataFrame(records).head(30))
else:
    print("Нет распознанных кропов.")
